# Latency survival and two-market movement analysis

This notebook measures what happens between detection and a delayed reprice. It separates the movement of the Binance leg from the movement of the Uniswap leg, which is essential: an opportunity survives when the **relative** move remains favorable, not merely when ETH rises or falls.

A survived signal is still a paper result. It does not model transaction propagation, builder ordering, partial CEX fills or failed/reverted on-chain execution.

In [ ]:
from pathlib import Path
import json
import os
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.style.use("seaborn-v0_8-whitegrid")


def find_repo_root(start=Path.cwd()):
    # Work whether Jupyter starts in the repository root or notebooks/.
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current directory")


ROOT = find_repo_root()
DB_PATH = Path(os.environ.get("ARB_DB_PATH", ROOT / "data" / "observations.sqlite3"))
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"No research database at {DB_PATH}. Run `python -m src --config config.toml monitor` first, "
        "or set ARB_DB_PATH to another observations.sqlite3 file."
    )

# Read-only URI prevents a notebook from accidentally changing the research dataset.
connection = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    connection,
)["name"].tolist()
print(f"Database: {DB_PATH}")
print("Tables:", ", ".join(tables))

## Join signals, delayed evaluations and both market snapshots

In [ ]:
query = '''
SELECT
    s.id AS signal_id, s.observation_id AS original_observation_id,
    s.created_at, s.due_at, s.delay_ms AS requested_delay_ms, s.direction, s.quantity_eth,
    s.expected_profit_usdc, s.status,
    e.evaluated_observation_id, e.evaluated_at, e.actual_delay_ms,
    e.repriced_profit_usdc, e.repriced_profit_bps, e.survived, e.error,
    original.binance_best_bid AS original_bid,
    original.binance_best_ask AS original_ask,
    original.uniswap_price AS original_dex,
    evaluated.binance_best_bid AS evaluated_bid,
    evaluated.binance_best_ask AS evaluated_ask,
    evaluated.uniswap_price AS evaluated_dex
FROM paper_signals s
LEFT JOIN latency_evaluations e ON e.signal_id = s.id
LEFT JOIN observations original ON original.id = s.observation_id
LEFT JOIN observations evaluated ON evaluated.id = e.evaluated_observation_id
ORDER BY s.id
'''
latency = pd.read_sql_query(query, connection)

numeric_columns = [
    "requested_delay_ms", "quantity_eth", "expected_profit_usdc", "actual_delay_ms",
    "repriced_profit_usdc", "repriced_profit_bps", "survived", "original_bid", "original_ask",
    "original_dex", "evaluated_bid", "evaluated_ask", "evaluated_dex",
]
for column in numeric_columns:
    latency[column] = pd.to_numeric(latency[column], errors="coerce")
for column in ["created_at", "due_at", "evaluated_at"]:
    latency[column] = pd.to_datetime(latency[column], utc=True, errors="coerce")

evaluated = latency[latency["evaluated_at"].notna()].copy()
coverage = pd.Series({
    "scheduled signals": len(latency),
    "evaluated signals": len(evaluated),
    "pending/unevaluated": len(latency) - len(evaluated),
    "evaluation coverage": len(evaluated) / max(len(latency), 1),
    "evaluation errors": int(evaluated["error"].notna().sum()) if len(evaluated) else 0,
})
coverage.to_frame("value")

## Requested versus actual delay

In [ ]:
if evaluated.empty:
    print("No delayed evaluations yet. Keep the monitor running long enough to reach configured delay horizons.")
else:
    evaluated["delay_overshoot_ms"] = evaluated["actual_delay_ms"] - evaluated["requested_delay_ms"]
    display(evaluated.groupby("requested_delay_ms").agg(
        samples=("signal_id", "size"),
        median_actual_ms=("actual_delay_ms", "median"),
        p95_actual_ms=("actual_delay_ms", lambda values: values.quantile(.95)),
        median_overshoot_ms=("delay_overshoot_ms", "median"),
    ))

    axes = evaluated.plot.scatter(
        x="requested_delay_ms", y="actual_delay_ms", alpha=.5, figsize=(7, 5),
        title="Requested delay vs actual observation-based reprice delay",
    )
    limit = max(evaluated["requested_delay_ms"].max(), evaluated["actual_delay_ms"].max())
    axes.plot([0, limit], [0, limit], "k--", linewidth=1, label="on time")
    axes.legend(); plt.show()

The monitor can only evaluate on a later observation. With a five-second polling interval, a requested 250 ms or 1 s horizon will generally be measured at the next poll, not at the exact requested timestamp. `actual_delay_ms` is therefore the correct x-axis for timing conclusions.

## Survival curve and profit decay

In [ ]:
if not evaluated.empty:
    evaluated["profit_change_usdc"] = evaluated["repriced_profit_usdc"] - evaluated["expected_profit_usdc"]
    by_horizon = evaluated.groupby("requested_delay_ms").agg(
        samples=("signal_id", "size"),
        survival_rate=("survived", "mean"),
        median_expected_profit=("expected_profit_usdc", "median"),
        median_repriced_profit=("repriced_profit_usdc", "median"),
        median_profit_change=("profit_change_usdc", "median"),
        p10_repriced_profit=("repriced_profit_usdc", lambda values: values.quantile(.10)),
    )
    display(by_horizon)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(by_horizon.index, by_horizon["survival_rate"], marker="o")
    axes[0].set(xlabel="Requested delay (ms)", ylabel="Survival probability", ylim=(-.05, 1.05), title="Opportunity survival")
    axes[1].plot(by_horizon.index, by_horizon["median_expected_profit"], marker="o", label="At detection")
    axes[1].plot(by_horizon.index, by_horizon["median_repriced_profit"], marker="o", label="After delay")
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set(xlabel="Requested delay (ms)", ylabel="Median profit (USDC)", title="Profit decay")
    axes[1].legend(); plt.show()

## Attribute the change to Binance and Uniswap

In [ ]:
if not evaluated.empty:
    evaluated["cex_leg_move_bps"] = 0.0
    evaluated["dex_leg_move_bps"] = 0.0
    buy_dex = evaluated["direction"].eq("buy_uniswap_sell_binance")
    buy_cex = evaluated["direction"].eq("buy_binance_sell_uniswap")

    # Positive means the sale venue improved or the purchase venue became cheaper.
    evaluated.loc[buy_dex, "cex_leg_move_bps"] = (
        evaluated.loc[buy_dex, "evaluated_bid"] / evaluated.loc[buy_dex, "original_bid"] - 1
    ) * 10_000
    evaluated.loc[buy_dex, "dex_leg_move_bps"] = -(
        evaluated.loc[buy_dex, "evaluated_dex"] / evaluated.loc[buy_dex, "original_dex"] - 1
    ) * 10_000

    evaluated.loc[buy_cex, "cex_leg_move_bps"] = -(
        evaluated.loc[buy_cex, "evaluated_ask"] / evaluated.loc[buy_cex, "original_ask"] - 1
    ) * 10_000
    evaluated.loc[buy_cex, "dex_leg_move_bps"] = (
        evaluated.loc[buy_cex, "evaluated_dex"] / evaluated.loc[buy_cex, "original_dex"] - 1
    ) * 10_000
    evaluated["relative_leg_move_bps"] = evaluated["cex_leg_move_bps"] + evaluated["dex_leg_move_bps"]

    display(evaluated.groupby(["requested_delay_ms", "survived"])[[
        "cex_leg_move_bps", "dex_leg_move_bps", "relative_leg_move_bps", "profit_change_usdc"
    ]].median())

    colors = evaluated["survived"].map({1: "tab:green", 0: "tab:red"})
    plt.figure(figsize=(7, 6))
    plt.scatter(evaluated["cex_leg_move_bps"], evaluated["dex_leg_move_bps"], c=colors, alpha=.55)
    plt.axhline(0, color="black", linewidth=.8); plt.axvline(0, color="black", linewidth=.8)
    plt.xlabel("Favorable Binance-leg move (bps)")
    plt.ylabel("Favorable Uniswap-leg move (bps)")
    plt.title("Both market movements determine opportunity survival")
    plt.show()

In the movement chart, upper/right movements help the original trade and lower/left movements hurt it. This direction-aware transformation avoids the mistake of calling a general ETH price rise favorable or unfavorable without considering which venue was the buy leg and which was the sell leg.

## Direction, size and delay risk

In [ ]:
if not evaluated.empty:
    display(evaluated.groupby(["direction", "requested_delay_ms"]).agg(
        samples=("signal_id", "size"),
        survival_rate=("survived", "mean"),
        median_quantity_eth=("quantity_eth", "median"),
        median_profit_change=("profit_change_usdc", "median"),
    ))

    evaluated["size_bucket"] = pd.qcut(evaluated["quantity_eth"], q=min(4, evaluated["quantity_eth"].nunique()), duplicates="drop")
    display(evaluated.groupby("size_bucket", observed=True).agg(
        samples=("signal_id", "size"), survival_rate=("survived", "mean"),
        median_expected_profit=("expected_profit_usdc", "median"),
        median_repriced_profit=("repriced_profit_usdc", "median"),
    ))

## Interpretation checklist

- Use actual delay, not only requested delay.
- Require enough samples per horizon before comparing survival rates.
- Inspect Binance and Uniswap movements separately; correlated ETH movement can hide a changing cross-venue basis.
- A delayed positive quote does not prove atomic execution or fillability.
- The monitor's polling cadence limits the resolution of this evaluation.